# LFM2 NL2Bash logit distillation (A100)

This notebook distills `Qwen/Qwen3.8-27B` into `LiquidAI/LFM2-350M`, then reloads the result in NF4/Q4 for evaluation. Select **Runtime → Change runtime type → A100 GPU** before starting. If an older revision of this notebook upgraded NumPy, use **Runtime → Disconnect and delete runtime** once before running this revision; Drive checkpoints are unaffected.

The models have incompatible vocabularies (65,536 vs. 248,320 rows). The training script therefore transplants the student's learned embedding and LM-head rows into the Qwen tokenizer wherever token strings match. This makes full-vocabulary KL valid. The notebook downloads the pre-quantized 22.3 GB NF4 teacher directly, so it never fetches the 55.6 GB BF16 checkpoint. The student's transformer body is unchanged, but its expanded vocabulary increases its parameter count.

The loss is response-only `0.3 × gold CE + 0.7 × KL(teacher || student)` at temperature 2.0. Teacher logits are computed online under the gold Bash trajectory, so no hundreds-of-gigabytes logit cache is needed.

In [ ]:
!nvidia-smi
# This is a text-only, full-weight run. Colab's preinstalled PEFT is not
# Transformers-v5 compatible, and a mismatched torchvision can break even
# text-model imports. Neither package is used here.
!pip -q uninstall -y peft torchvision
!pip -q install --upgrade -r https://raw.githubusercontent.com/micrictor/shellai/training/requirements-distill.txt
!python -c "import numpy, torch, transformers; print('numpy', numpy.__version__, '| torch', torch.__version__, 'cuda', torch.version.cuda, '| transformers', transformers.__version__)"

In [ ]:
import codecs, errno, os, pty, select, shlex, subprocess, time
from pathlib import Path
from google.colab import drive, userdata

COMMAND_RESULTS = []

def run_command(command, **popen_kwargs):
    command = [str(part) for part in command]
    print(f'\n$ {shlex.join(command)}', flush=True)
    child_env = dict(os.environ)
    child_env['PYTHONUNBUFFERED'] = '1'
    child_env.update(popen_kwargs.pop('env', {}))
    # A PTY makes tqdm/Hugging Face detect a real terminal. Reading raw
    # chunks also renders carriage-return progress updates immediately.
    master_fd, slave_fd = pty.openpty()
    try:
        process = subprocess.Popen(
            command, stdout=slave_fd, stderr=slave_fd, env=child_env, **popen_kwargs
        )
    finally:
        os.close(slave_fd)
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    output_chunks = []
    started = last_activity = time.monotonic()
    try:
        while True:
            ready, _, _ = select.select([master_fd], [], [], 1.0)
            if ready:
                try:
                    data = os.read(master_fd, 4096)
                except OSError as error:
                    if error.errno == errno.EIO:  # Normal PTY EOF on Linux.
                        break
                    raise
                if not data:
                    break
                text = decoder.decode(data)
                output_chunks.append(text)
                print(text, end='', flush=True)
                last_activity = time.monotonic()
            elif process.poll() is not None:
                break
            elif time.monotonic() - last_activity >= 30:
                elapsed = int(time.monotonic() - started)
                print(f'\n[heartbeat] command still running ({elapsed}s elapsed)', flush=True)
                last_activity = time.monotonic()
    finally:
        os.close(master_fd)
    remainder = decoder.decode(b'', final=True)
    if remainder:
        output_chunks.append(remainder)
        print(remainder, end='', flush=True)
    return_code = process.wait()
    combined_output = ''.join(output_chunks)
    result = subprocess.CompletedProcess(
        command, return_code, stdout=combined_output, stderr=None
    )
    COMMAND_RESULTS.append(result)
    if return_code != 0:
        output_lines = combined_output.splitlines()
        tail_lines = output_lines[-200:]
        tail = '\n'.join(tail_lines)
        raise RuntimeError(
            f'Command failed with exit code {return_code}: {shlex.join(command)}\n'
            f'Last {len(tail_lines)} output lines:\n{tail}'
        ) from None
    return result

drive.mount('/content/drive')
repo = Path('/content/shellai')
if not repo.exists():
    run_command(['git', 'clone', '--branch', 'training', '--single-branch', 'https://github.com/micrictor/shellai.git', str(repo)])
else:
    run_command(['git', '-C', str(repo), 'pull', '--ff-only'])
os.chdir(repo)

# Optional: create a Colab secret named HF_TOKEN for authenticated downloads
# and another named HF_REPO_ID (for example, username/shellai-lfm2-qwen38-nl2bash-distilled)
# if you want the final checkpoint pushed to the Hub.
try:
    token = userdata.get('HF_TOKEN')
except Exception:
    token = None
if token:
    os.environ['HF_TOKEN'] = token
print('Working directory:', Path.cwd())

# Fetch the pre-quantized NF4 teacher before launching the trainer. Doing this
# notebook process gives Hugging Face's per-file progress bars; otherwise
# the trainer can appear to emit only heartbeats for many minutes. Partial
# downloads resume if this cell is interrupted and rerun in the same runtime.
import shutil
from huggingface_hub import snapshot_download
from huggingface_hub.utils import enable_progress_bars
enable_progress_bars()

TEACHER_REPO = 'unsloth/Qwen3.8-27B-unsloth-bnb-4bit'
TEACHER_REVISION = 'b0fff092a4c8ec2c050e8e3d301bf6513bea4048'
free_gib = shutil.disk_usage('/content').free / (1024 ** 3)
cache_blobs = Path.home() / '.cache/huggingface/hub/models--unsloth--Qwen3.8-27B-unsloth-bnb-4bit/blobs'
cached_gib = sum(path.stat().st_size for path in cache_blobs.glob('*') if path.is_file()) / (1024 ** 3)
remaining_gib = max(0.0, 23.0 - cached_gib)
print(f'Free disk: {free_gib:.1f} GiB | teacher already cached: {cached_gib:.1f} GiB', flush=True)
if free_gib < remaining_gib + 5:
    raise RuntimeError(f'Teacher needs about {remaining_gib:.1f} GiB more plus working space; only {free_gib:.1f} GiB is free.')
print(f'Downloading/caching the 22.3 GB NF4 teacher {TEACHER_REPO}...', flush=True)
TEACHER_MODEL = snapshot_download(
    repo_id=TEACHER_REPO, revision=TEACHER_REVISION, token=token, max_workers=8,
    allow_patterns=[
        'config.json', 'generation_config.json', 'model*.safetensors*',
        'tokenizer*', 'chat_template.jinja', 'vocab.json', 'merges.txt',
        'special_tokens_map.json',
    ],
)
teacher_bytes = sum(path.stat().st_size for path in Path(TEACHER_MODEL).rglob('*') if path.is_file())
print(f'Teacher ready: {TEACHER_MODEL} ({teacher_bytes / (1024 ** 3):.1f} GiB)', flush=True)

## 1. Smoke test

This uses only 64 training examples. It verifies model loading, vocabulary transplantation, forward/backward passes, checkpoint writing, and validation before committing an A100 session to the full run.

In [ ]:
RUN_SMOKE_TEST = True

if RUN_SMOKE_TEST:
    smoke_command = [
        'python', 'train_lfm_distill.py',
        '--teacher-model', TEACHER_MODEL,
        '--output-dir', '/content/lfm-distill-smoke',
        '--epochs', '1',
        '--max-train-samples', '64',
        '--max-validation-samples', '32',
        '--batch-size', '1',
        '--gradient-accumulation-steps', '8',
        '--logging-steps', '2',
        '--no-save-every-epoch',
    ]
    smoke_result = run_command(smoke_command)

## 2. Full distillation run

Defaults: two epochs, micro-batch 1, gradient accumulation 16 (effective batch 16), and 256-token sequences. `checkpoint-last` is replaced after each epoch so optimizer state does not consume all of Drive; `final` is a clean inference checkpoint. The teacher is loaded in NF4; exact full-vocabulary logits are still distilled online. If a 40 GB A100 runs out of memory, reduce `max-length` to 192 for the smoke test before changing the full run. The official 300-row NL2SH-ALFA test configuration is never loaded during training.

In [ ]:
RUN_FULL_TRAINING = True
OUTPUT_DIR = '/content/drive/MyDrive/shellai/lfm2-qwen38-nl2bash-distilled'
RESUME_FROM = None  # set to OUTPUT_DIR + '/checkpoint-last' after an interruption

if RUN_FULL_TRAINING:
    command = [
        'python', 'train_lfm_distill.py',
        '--teacher-model', TEACHER_MODEL,
        '--output-dir', OUTPUT_DIR,
        '--epochs', '2',
        '--batch-size', '1',
        '--gradient-accumulation-steps', '16',
        '--max-length', '256',
        '--temperature', '2.0',
        '--distill-weight', '0.7',
        '--learning-rate', '2e-5',
        '--logging-steps', '10',
    ]
    if RESUME_FROM:
        command += ['--resume-from', RESUME_FROM]
    try:
        hub_repo = userdata.get('HF_REPO_ID')
    except Exception:
        hub_repo = None
    if hub_repo:
        command += ['--push-to-hub', hub_repo]
    print(' '.join(command))
    training_result = run_command(command)

## 3. Q4 evaluation against the original student

This runtime-quantizes each model with bitsandbytes NF4, evaluates deterministic generation on the isolated official test set, and unloads one model before loading the next. It never executes generated commands. Exact match is strict; token F1 and simple syntax balance are more informative during iteration.

In [ ]:
import gc, re, time
from collections import Counter
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

SYSTEM = ('You translate natural-language requests into exactly one Bash command. '
          'Return only the command, with no Markdown, explanation, or alternatives.')

def make_prompt(nl, tokenizer):
    return tokenizer.apply_chat_template(
        [
            {'role': 'system', 'content': SYSTEM},
            {'role': 'user', 'content': 'Generate one Bash command: ' + nl.strip()},
        ],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )

def clean_answer(text):
    text = text.split('<|im_end|>', 1)[0].strip()
    fenced = re.search(r'```(?:bash|sh)?\s*(.*?)```', text, re.I | re.S)
    return (fenced.group(1) if fenced else text).strip()

def token_f1(reference, candidate):
    ref, pred = reference.split(), candidate.split()
    if not ref or not pred: return 0.0
    overlap = sum((Counter(ref) & Counter(pred)).values())
    if not overlap: return 0.0
    p, r = overlap / len(pred), overlap / len(ref)
    return 2 * p * r / (p + r)

def syntax_balanced(command):
    return (bool(command) and command.count(chr(34)) % 2 == 0
            and command.count(chr(39)) % 2 == 0
            and command.count('(') == command.count(')')
            and command.count('[') == command.count(']'))

def evaluate_q4(model_id, samples):
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                  bnb_4bit_use_double_quant=True,
                                  bnb_4bit_compute_dtype=torch.bfloat16)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=quant, device_map={'': 0}, dtype=torch.bfloat16)
    predictions, latencies = [], []
    for row in samples:
        encoded = tokenizer(make_prompt(row['nl'], tokenizer), return_tensors='pt').to('cuda')
        torch.cuda.synchronize(); started = time.perf_counter()
        with torch.inference_mode():
            generated = model.generate(**encoded, max_new_tokens=96, do_sample=False,
                                       eos_token_id=tokenizer.eos_token_id,
                                       pad_token_id=tokenizer.pad_token_id)
        torch.cuda.synchronize(); latencies.append(1000 * (time.perf_counter() - started))
        continuation = generated[0, encoded['input_ids'].shape[1]:]
        predictions.append(clean_answer(tokenizer.decode(continuation, skip_special_tokens=False)))
    references = [[row['bash']] + ([row['bash2']] if row.get('bash2') else []) for row in samples]
    result = {
        'model': str(model_id),
        'examples': len(samples),
        'exact_match_pct': 100 * np.mean([p in rs for p, rs in zip(predictions, references)]),
        'token_f1': np.mean([max(token_f1(r, p) for r in rs) for p, rs in zip(predictions, references)]),
        'syntax_balanced_pct': 100 * np.mean([syntax_balanced(p) for p in predictions]),
        'latency_ms': np.mean(latencies),
    }
    del model, tokenizer
    gc.collect(); torch.cuda.empty_cache()
    return result, predictions[:5]

test = load_dataset('westenfelder/NL2SH-ALFA', 'test', split='train').select(range(100))
results = []
for model_id in ['LiquidAI/LFM2-350M', OUTPUT_DIR + '/final']:
    result, examples = evaluate_q4(model_id, test)
    results.append(result)
    print(result)
    print('First five predictions:', examples)
results

## Reading the result

Do not select a checkpoint from training loss alone. Prefer lower held-out KL **and** improved official-test token F1/exact match after Q4 loading. If the student imitates teacher quirks while command accuracy falls, rerun with `--distill-weight 0.5`. If both validation CE and KL are still falling at epoch 2, resume for one additional epoch at `1e-5`. Keep the official test set out of all such choices; for repeated tuning, use the validation split and run the official test once on the chosen configuration.